In [ ]:
import pandas as pd
import numpy as np
import requests
from pathlib import Path
from bs4 import BeautifulSoup
import unicodedata

# ── Change this one line if your CSVs live elsewhere ──────────────────────────
DATA_DIR = Path.home() / "Downloads"

# ── Fetch Wikipedia squad page ONCE; reuse html everywhere ───────────────────
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                  "AppleWebKit/537.36 (KHTML, like Gecko) "
                  "Chrome/124.0.0.0 Safari/537.36"
}
url  = "https://en.wikipedia.org/wiki/2022_FIFA_World_Cup_squads"
html = requests.get(url, headers=headers).text
tables = pd.read_html(html)
print(f"Total tables found: {len(tables)}")

In [ ]:
squad_tables = []
for t in tables:
    cols = [str(c).lower() for c in t.columns]
    if any(kw in cols for kw in ["player", "pos.", "date of birth (age)"]):
        squad_tables.append(t)
print(f"Squad tables: {len(squad_tables)}")

In [ ]:
soup = BeautifulSoup(html, "html.parser")   # reuses html from Cell 0

team_names = []
for header in soup.find_all(["h2", "h3"]):
    text = header.get_text(strip=True).replace("[edit]", "")
    if "Group" in text or "Squads" in text:
        continue
    if header.find_next("table", {"class": "wikitable"}):
        team_names.append(text)

print(team_names)
len(team_names)

In [ ]:
teams_to_drop = [
    'Contents', 'Player representation by club', 
    'Player representation by club confederation', 
    'Average age of squads', 'Coaches representation by country','Statistics', 'Age', 'Players', 'Player representation by league system']

team_names = [n for n in team_names if n not in teams_to_drop]
print(team_names)
print(f"{len(team_names)} teams, {len(squad_tables)} squad tables")


assert len(team_names) == len(squad_tables), \
    f"Mismatch: {len(team_names)} names vs {len(squad_tables)} tables. Check scraping."

clean_squads = []
for team, table in zip(team_names, squad_tables):
    df = table.copy()
    df.columns = df.columns.str.lower()
    df["team"] = team
    clean_squads.append(df)

squads_df = pd.concat(clean_squads, ignore_index=True)
print(squads_df.shape)
squads_df.head()

In [ ]:
%%html
<style>
#notebook-container { width: 98% !important; }
div.output_scroll { height: unset !important; }
div.output_area pre { white-space: pre; overflow-x: auto; }
table.dataframe { display: block; overflow-x: auto; white-space: nowrap; }
</style>

In [ ]:
players_df  = pd.read_csv(DATA_DIR / "players.csv")
valuations  = pd.read_csv(DATA_DIR / "player_valuations.csv")

valuations["date"] = pd.to_datetime(valuations["date"])

# Latest market value per player
latest_vals = (
    valuations[valuations["date"] <= "2022-11-20"] 
    .sort_values("date")
    .groupby("player_id")
    .tail(1)
    .reset_index(drop=True)
)


players_df_clean = players_df.drop(columns=["market_value_in_eur"], errors="ignore")
players_latest = players_df_clean.merge(
    latest_vals[["player_id", "market_value_in_eur"]],
    on="player_id",
    how="left"
)

print(f"Missing market values: {players_df['market_value_in_eur'].isna().sum()}")
print(players_latest.columns.tolist())

In [ ]:
def normalize(s):
    s = unicodedata.normalize("NFKD", str(s))
    s = s.encode("ascii", "ignore").decode("utf-8")
    return s.lower().strip()

squads_df["player_clean"]     = squads_df["player"].apply(normalize)
players_latest["player_clean"] = players_latest["name"].apply(normalize)

players_deduped = (
    players_latest
    .sort_values("market_value_in_eur", ascending=False)   # FIX: no _y suffix
    .drop_duplicates(subset="player_clean", keep="first")
)

merged = squads_df.merge(
    players_deduped[["player_clean", "market_value_in_eur"]],   # FIX: no _y
    on="player_clean",
    how="left"
)
print(merged.shape)
merged.head()

In [ ]:
print("Unique players:", merged["player_clean"].nunique())
merged[merged.duplicated(subset="player_clean", keep=False)][["player_clean", "team"]].head(20)

In [ ]:
results = pd.read_csv(DATA_DIR / "results.csv")
results["date"] = pd.to_datetime(results["date"])

wc_teams = squads_df["team"].unique()


results_filtered = results[
    (results["date"] >= "2010-01-01") & 
    (results["date"] < "2022-11-20") & # <── ADD THIS UPPER BOUND
    (results["home_team"].isin(wc_teams) | results["away_team"].isin(wc_teams))
]

keep_tournaments = [
    "FIFA World Cup", "FIFA World Cup qualification",
    "UEFA Euro qualification", "UEFA Euro",
    "African Cup of Nations qualification", "African Cup of Nations",
    "Gold Cup", "Copa América", "Copa América qualification",
    "AFC Asian Cup", "AFC Asian Cup qualification",
    "UEFA Nations League", "CONCACAF Nations League",
    "Confederations Cup",
]

results_competitive = results_filtered[
    results_filtered["tournament"].isin(keep_tournaments)
].copy()

print(results_competitive.shape)

wc_teams_set  = set(wc_teams)
results_teams = set(results_competitive["home_team"]) | set(results_competitive["away_team"])
missing_teams = wc_teams_set - results_teams
if missing_teams:
    print("WARNING — teams with no competitive results (will get neutral stats):", missing_teams)

In [ ]:
def compute_team_stats(team, df, since="2014-01-01"):
    matches = df[
        ((df["home_team"] == team) | (df["away_team"] == team)) &
        (df["date"] >= since)
    ]

    # FIX: guard against zero matches to avoid division by zero
    if len(matches) == 0:
        return {
            "team": team, "matches": 0,
            "win_rate": 0.33, "draw_rate": 0.33, "loss_rate": 0.33,
            "avg_goals_scored": 1.0, "avg_goals_conceded": 1.0,
            "goal_difference": 0.0,
        }

    rows = []
    for _, row in matches.iterrows():
        if row["home_team"] == team:
            gf, ga = row["home_score"], row["away_score"]
        else:
            gf, ga = row["away_score"], row["home_score"]
        result = "W" if gf > ga else ("D" if gf == ga else "L")
        rows.append({"gf": gf, "ga": ga, "result": result})

    rdf = pd.DataFrame(rows)
    n   = len(rdf)
    return {
        "team":               team,
        "matches":            n,
        "win_rate":           (rdf["result"] == "W").sum() / n,
        "draw_rate":          (rdf["result"] == "D").sum() / n,
        "loss_rate":          (rdf["result"] == "L").sum() / n,
        "avg_goals_scored":   rdf["gf"].mean(),
        "avg_goals_conceded": rdf["ga"].mean(),
        "goal_difference":    (rdf["gf"] - rdf["ga"]).mean(),
    }

team_stats = pd.DataFrame([compute_team_stats(t, results_competitive) for t in wc_teams])
team_stats = team_stats.sort_values("win_rate", ascending=False)
print(team_stats.head(20))

In [ ]:
games       = pd.read_csv(DATA_DIR / "games.csv")
appearances = pd.read_csv(DATA_DIR / "appearances.csv")
clubs       = pd.read_csv(DATA_DIR / "clubs.csv")
valuations  = pd.read_csv(DATA_DIR / "player_valuations.csv")

valuations["date"] = pd.to_datetime(valuations["date"])
games["date"]      = pd.to_datetime(games["date"])

print("games:",       games.shape)
print("appearances:", appearances.shape)
print("clubs:",       clubs.shape)
print("valuations:",  valuations.shape)
print("\ngames columns:",       games.columns.tolist())
print("appearances columns:", appearances.columns.tolist())

In [ ]:
print(games["competition_type"].value_counts())

In [ ]:
# ── Prime score function (Branquinho et al. Gaussian) ────────────────────────
def calculate_prime_score(age):
    if pd.isna(age) or age <= 0:
        return 0.7          # Conservative baseline for missing/invalid ages
    PEAK_AGE = 25.5         # Midpoint of endurance (24.8) & explosive (26.0) peaks
    SIGMA    = 3.5          # Gradual rise, sharp drop after 32
    return np.exp(-((age - PEAK_AGE) ** 2) / (2 * SIGMA ** 2))

# Step 1 — domestic league games from 2005 onwards
league_games = games[
    (games["competition_type"] == "domestic_league") &
    (games["date"] >= "2005-01-01") &
    (games["date"] < "2022-11-20") 
].copy()
print("league_games:", league_games.shape)

# Step 2 — Filter appearances to match selected games 

league_app = appearances[appearances["game_id"].isin(league_games["game_id"])].copy()
league_app["date"] = pd.to_datetime(league_app["date"])
print("league_app:", league_app.shape)

# Step 3 — merge with valuations; keep only valuations on/before match date
historic_valuations = valuations[valuations["date"] <= "2022-11-20"].copy()
historic_valuations["date_val"] = pd.to_datetime(historic_valuations["date"])

league_app_vals = league_app.merge(
    historic_valuations[["player_id", "date_val", "market_value_in_eur"]],
    on="player_id",
    how="left"
)

# Keep rows that are valid historical records OR rows where no valuation ever existed
league_app_vals = league_app_vals[
    (league_app_vals["date_val"] <= league_app_vals["date"]) | 
    (league_app_vals["date_val"].isna())
]

# Sort and get the most recent valuation prior to the match day
league_app_vals = (
    league_app_vals
    .sort_values("date_val")
    .groupby("appearance_id")
    .tail(1)
).copy()

# Fill missing market values with a default baseline so calculations don't break
league_app_vals["market_value_in_eur"] = league_app_vals["market_value_in_eur"].fillna(50_000)

print("After value join:", league_app_vals.shape)
print(league_app_vals["market_value_in_eur"].isna().sum(), "missing market values")

# Step 4 — attach DOB and compute match age + prime score
players_dob = players_df[["player_id", "date_of_birth"]].copy()
players_dob["date_of_birth"] = pd.to_datetime(players_dob["date_of_birth"])
league_app_vals = league_app_vals.merge(players_dob, on="player_id", how="left")

league_app_vals["match_age"] = (
    league_app_vals["date"] - league_app_vals["date_of_birth"]
).dt.days / 365.25

# Performance vectorization calculation loop
age_array = league_app_vals["match_age"].values
PEAK_AGE = 25.5
SIGMA = 3.5

prime_scores = np.exp(-((age_array - PEAK_AGE) ** 2) / (2 * SIGMA ** 2))
prime_scores[(age_array <= 0) | (np.isnan(age_array))] = 0.7 
league_app_vals["prime_score"] = prime_scores

print("Age + prime score attached.")

In [ ]:
def gini(values):
    values = sorted(values)
    n = len(values)
    if n == 0 or sum(values) == 0:
        return 0
    cumsum = sum((2 * (i + 1) - n - 1) * v for i, v in enumerate(values))
    return cumsum / (n * sum(values))

squad_values = (
    league_app_vals
    .groupby(["game_id", "player_club_id"])
    .agg(
        total_value    =("market_value_in_eur", "sum"),
        avg_value      =("market_value_in_eur", "mean"),
        num_players    =("market_value_in_eur", "count"),
        avg_prime_score=("prime_score",          "mean"),
    )
    .reset_index()
)

gini_vals = (
    league_app_vals
    .groupby(["game_id", "player_club_id"])["market_value_in_eur"]
    .apply(gini)
    .reset_index()
    .rename(columns={"market_value_in_eur": "gini"})
)

squad_values = squad_values.merge(gini_vals, on=["game_id", "player_club_id"])
print(squad_values.shape)
squad_values.head()

In [ ]:
records = []
for _, row in league_games.iterrows():
    records.append({
        "game_id": row["game_id"], "date": row["date"],
        "club_id": row["home_club_id"],
        "win": 1 if row["home_club_goals"] > row["away_club_goals"] else 0,
    })
    records.append({
        "game_id": row["game_id"], "date": row["date"],
        "club_id": row["away_club_id"],
        "win": 1 if row["away_club_goals"] > row["home_club_goals"] else 0,
    })

form_records = pd.DataFrame(records).sort_values(["club_id", "date"])
form_records["form"] = (
    form_records.groupby("club_id")["win"]
    .transform(lambda x: x.shift(1).rolling(5, min_periods=1).mean())
)
form_records["form"] = form_records["form"].fillna(0.5)
print(form_records.shape)
form_records.head(10)

In [ ]:
home_form = form_records[["game_id", "club_id", "form"]].rename(
    columns={"club_id": "home_club_id", "form": "home_form"})
away_form = form_records[["game_id", "club_id", "form"]].rename(
    columns={"club_id": "away_club_id", "form": "away_form"})


home = games[["game_id", "home_club_id", "home_club_goals"]].merge(
    squad_values.rename(columns={
        "player_club_id":  "home_club_id",
        "total_value":     "home_total_value",
        "avg_value":       "home_avg_value",
        "gini":            "home_gini",
        "num_players":     "home_num_players",
        "avg_prime_score": "home_avg_prime_score",
    }),
    on=["game_id", "home_club_id"]
)

away = games[["game_id", "away_club_id", "away_club_goals"]].merge(
    squad_values.rename(columns={
        "player_club_id":  "away_club_id",
        "total_value":     "away_total_value",
        "avg_value":       "away_avg_value",
        "gini":            "away_gini",
        "num_players":     "away_num_players",
        "avg_prime_score": "away_avg_prime_score",  # FIX: was missing before
    }),
    on=["game_id", "away_club_id"]
)


train_df = home.merge(
    away[["game_id", "away_club_id", "away_club_goals", "away_total_value", 
          "away_avg_value", "away_gini", "away_num_players", "away_avg_prime_score"]],
    on="game_id"
)

train_df = train_df.merge(home_form, on=["game_id", "home_club_id"], how="left")
train_df = train_df.merge(away_form, on=["game_id", "away_club_id"], how="left")

train_df["home_form"] = train_df["home_form"].fillna(0.5)
train_df["away_form"] = train_df["away_form"].fillna(0.5)

train_df["result"] = train_df.apply(
    lambda r: 1 if r["home_club_goals"] > r["away_club_goals"]
    else (0 if r["home_club_goals"] == r["away_club_goals"] else -1), axis=1
)

train_df["total_value_diff"] = train_df["home_total_value"]     - train_df["away_total_value"]
train_df["avg_value_diff"]   = train_df["home_avg_value"]       - train_df["away_avg_value"]
train_df["gini_diff"]        = train_df["home_gini"]            - train_df["away_gini"]
train_df["form_diff"]        = train_df["home_form"]            - train_df["away_form"]
train_df["prime_diff"]       = train_df["home_avg_prime_score"] - train_df["away_avg_prime_score"]

print(train_df.shape)
print(train_df["result"].value_counts())

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

train_df['total_value_diff'] = train_df['total_value_diff'] /1e9
train_df['avg_value_diff'] = train_df['avg_value_diff'] / 1e7


features = ["total_value_diff", "avg_value_diff", "gini_diff", "form_diff", "prime_diff"]

X = train_df[features]

y = train_df["result"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = RandomForestClassifier(n_estimators=200, random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

importances = pd.DataFrame({
    "feature":    features,
    "importance": model.feature_importances_,
}).sort_values("importance", ascending=False)
print(importances)

In [ ]:

merged["market_value_in_eur"] = merged["market_value_in_eur"].fillna(0)


# Extract age from Wikipedia's 'date of birth (age)' column safely
dob_col = next((c for c in merged.columns if "date of birth" in c.lower() or "age" in c.lower()), None)

if dob_col:
    # Captures digits inside parentheses even if preceded by "aged " or "age "
    extracted_age = merged[dob_col].astype(str).str.extract(r"\((?:aged\s+|age\s+)?(\d+)\)")[0]
    
    # Cast to float, and fill missing values with a realistic default squad mean (26)
    merged["age"] = pd.to_numeric(extracted_age, errors='coerce').fillna(26.0)
elif "age" not in merged.columns:
    merged["age"] = 26.0  # Realistic baseline fallback for international players

merged["prime_score"] = merged["age"].apply(calculate_prime_score)

team_features = (
    merged.groupby("team")
    .agg(
        total_value    =("market_value_in_eur", "sum"),
        avg_value      =("market_value_in_eur", "mean"),
        num_players    =("market_value_in_eur", "count"),
        avg_prime_score=("prime_score",          "mean"),
        gini           =("market_value_in_eur", gini),
    )
    .reset_index()
)
print(team_features.sort_values("total_value", ascending=False).head(10))

# Check for any WC teams missing from team_features
groups_check = {
    "A": ["Qatar", "Ecuador", "Senegal", "Netherlands"],
    "B": ["England", "Iran", "United States", "Wales"],
    "C": ["Argentina", "Saudi Arabia", "Mexico", "Poland"],
    "D": ["France", "Australia", "Denmark", "Tunisia"],
    "E": ["Spain", "Costa Rica", "Germany", "Japan"],
    "F": ["Belgium", "Canada", "Morocco", "Croatia"],
    "G": ["Brazil", "Serbia", "Switzerland", "Cameroon"],
    "H": ["Portugal", "Ghana", "Uruguay", "South Korea"],
}
all_teams   = [t for teams in groups_check.values() for t in teams]
known_teams = set(team_features["team"])
missing     = [t for t in all_teams if t not in known_teams]
print("Missing from team_features:", missing)

In [ ]:



def compute_international_form_fixed(team, df, n=20, max_date="2022-11-20"):
    # Filter to only matches BEFORE the World Cup started to avoid leakage
    matches = df[((df["home_team"] == team) | (df["away_team"] == team)) & (df["date"] < max_date)]
    matches = matches.sort_values("date").tail(n)
    
    if len(matches) == 0:
        return 0.5
        
    form_score = 0
    for _, row in matches.iterrows():
        if row["home_team"] == team:
            team_goals, opp_goals = row["home_score"], row["away_score"]
        else:
            team_goals, opp_goals = row["away_score"], row["home_score"]
            
        
        if team_goals > opp_goals:
            form_score += 1.0
        elif team_goals == opp_goals:
            form_score += 0.5
            
    return form_score / len(matches)

# 2. Re-calculate form for all World Cup teams using the fixed logic
wc_teams = team_features["team"].unique()

int_form_fixed = {
    team: compute_international_form_fixed(team, results_competitive, n=20, max_date="2022-11-20")
    for team in wc_teams
}

# 3. Map the clean scores back to your main feature matrix
team_features["int_form"] = team_features["team"].map(int_form_fixed).fillna(0.5)

# 4. PRINT THE NEW FORM LEADERBOARD
print("PRE-WORLD CUP 2022 FORM")
print(team_features[["team", "int_form"]].sort_values("int_form", ascending=False).head(15).to_string(index=False))

In [ ]:
import pandas as pd
import numpy as np

# 1. 2022 Qatar World Cup Odds Dictionary 
odds_qatar2022 = {
    "Brazil": 350,
    "Argentina": 500,
    "England": 700,
    "France": 700,
    "Spain": 800,
    "Germany": 1000,
    "Netherlands": 1400,
    "Portugal": 1400,
    "Belgium": 1600,
    "Denmark": 2800,
    "Croatia": 5000,
    "Uruguay": 5000,
    "Serbia": 6600,
    "Senegal": 8000,
    "Switzerland": 8000,
    "Mexico": 10000,
    "United States": 10000,
    "Poland": 10000,
    "Wales": 10000,
    "Canada": 15000,
    "Ecuador": 15000,
    "Ghana": 15000,
    "Morocco": 20000,
    "Cameroon": 25000,
    "Japan": 25000,
    "Qatar": 25000,
    "South Korea": 30000,
    "Tunisia": 30000,
    "Australia": 40000,
    "Costa Rica": 50000,
    "Saudi Arabia": 50000,  # Fixed
    "Iran": 50000           # Fixed
}

# 2. normalization logic to transform American odds into true probabilities
def american_to_prob(o):
    return 100 / (o + 100) if o > 0 else abs(o) / (abs(o) + 100)

raw_probs = {t: american_to_prob(o) for t, o in odds_qatar2022.items()}
total_sum = sum(raw_probs.values())
odds_prob = {t: p / total_sum for t, p in raw_probs.items()}


# 3. Predict Match Function (Without Physics Layers)
def predict_match(team_a, team_b, venue="Neutral", odds_weight=0.6):
    a = team_features[team_features["team"] == team_a].iloc[0]
    b = team_features[team_features["team"] == team_b].iloc[0]

    
    X = pd.DataFrame([{
        "total_value_diff": (a["total_value"] - b["total_value"]) / 1e9,
        "avg_value_diff":   (a["avg_value"]   - b["avg_value"]) / 1e7,
        "gini_diff":        a["gini"]            - b["gini"],
        "form_diff":        a["int_form"]        - b["int_form"],
        "prime_diff":       a["avg_prime_score"] - b["avg_prime_score"],
    }])

    proba       = model.predict_proba(X)[0]
    classes     = model.classes_
    model_probs = {int(c): p for c, p in zip(classes, proba)}

    # Binary match-up probabilities extracted from the odds dictionary
    prob_a     = odds_prob.get(team_a, 0.01)
    prob_b     = odds_prob.get(team_b, 0.01)
    odds_win_a = prob_a / (prob_a + prob_b)
    odds_win_b = prob_b / (prob_a + prob_b)

    # Directly pull model outcomes 
    p_win_a = model_probs[1]
    p_win_b = model_probs[-1]
    p_draw  = model_probs[0]

    # Blending the machine learning model outcomes with the betting odds anchors
    blended_win_a = (1 - odds_weight) * p_win_a + odds_weight * odds_win_a
    blended_win_b = (1 - odds_weight) * p_win_b + odds_weight * odds_win_b
    blended_draw  = (1 - odds_weight) * p_draw  + odds_weight * model_probs[0]

    total = blended_win_a + blended_win_b + blended_draw
    return {
         1: blended_win_a / total,
         0: blended_draw  / total,
        -1: blended_win_b / total,
    }

In [ ]:
from itertools import combinations
from collections import Counter







# ── Build matchup cache 
from itertools import combinations


all_wc_teams = [t for teams in groups_check.values() for t in teams]
matchup_cache = {}

# Phase 1: Group stage combinations 
for grp, teams in groups_check.items():
    for h, aw in combinations(teams, 2):
        for a, b in [(h, aw), (aw, h)]:
            key = (str(a), str(b))
            if key not in matchup_cache:
                matchup_cache[key] = predict_match(str(a), str(b))

# Phase 2: Knockout combinations (All pairwise pairings across all 32 teams)
for ta, tb in combinations(all_wc_teams, 2):
    for a, b in [(ta, tb), (tb, ta)]:
        key = (str(a), str(b))
        if key not in matchup_cache:
            matchup_cache[key] = predict_match(str(a), str(b))

print("Cache built:", len(matchup_cache), "matchups")

In [ ]:
from itertools import combinations

def get_lambdas(team_a, team_b):
    a = team_features[team_features["team"] == team_a].iloc[0]
    b = team_features[team_features["team"] == team_b].iloc[0]
    val_diff = (a["total_value"] - b["total_value"]) / 1e9
    return max(0.5, 1.5 + val_diff), max(0.5, 1.5 - val_diff)


def simulate_score(lambda_a, lambda_b, outcome, max_tries=500):
    """Rejection sampling — no clamping distortion."""
    for _ in range(max_tries):
        ga = np.random.poisson(lambda_a)
        gb = np.random.poisson(lambda_b)
        if outcome == 1  and ga > gb: return ga, gb
        if outcome == -1 and gb > ga: return ga, gb
        if outcome == 0  and ga == gb: return ga, gb
    # Rare fallback
    return {1: (1, 0), -1: (0, 1), 0: (0, 0)}[outcome]


def simulate_group(teams):
    points = {t: 0 for t in teams}
    gd     = {t: 0 for t in teams}
    gf     = {t: 0 for t in teams}

    for home, away in combinations(teams, 2):
        
        
        proba   = matchup_cache[(str(home).strip(), str(away).strip())]
        outcome = np.random.choice([1, 0, -1], p=[proba[1], proba[0], proba[-1]])

        lh, la          = get_lambdas(home, away)
        goals_h, goals_a = simulate_score(lh, la, outcome)

        if outcome == 1:
            points[home] += 3
        elif outcome == -1:
            points[away] += 3
        else:
            points[home] += 1
            points[away] += 1

        gf[home] += goals_h;  gf[away] += goals_a
        gd[home] += goals_h - goals_a
        gd[away] += goals_a - goals_h

    standings = sorted(teams, key=lambda t: (points[t], gd[t], gf[t]), reverse=True)
    return [(t, points[t], gd[t], gf[t]) for t in standings]


def simulate_knockout(team_a, team_b):
    
    proba   = matchup_cache[(str(team_a), str(team_b))]
    outcome = np.random.choice([1, 0, -1], p=[proba[1], proba[0], proba[-1]])

    la, lb           = get_lambdas(team_a, team_b)
    simulate_score(la, lb, outcome)   # generates realistic score (not used for advancement)

    if outcome == 1:  return str(team_a)
    if outcome == -1: return str(team_b)
    return str(team_a) if np.random.random() < 0.5 else str(team_b)  # penalties




In [ ]:
def simulate_tournament():
    # ── 1. GROUP STAGE ──
    group_winners = {}
    group_runners_up = {}
    
    for g_id, teams in groups_check.items():
        table = simulate_group(teams)
        group_winners[g_id] = table[0][0]
        group_runners_up[g_id] = table[1][0]

    # ── 2. ROUND OF 16 ──
    r16_winners = [
        simulate_knockout(group_winners["A"], group_runners_up["B"]), # Ned vs USA
        simulate_knockout(group_winners["C"], group_runners_up["D"]), # Arg vs Aus
        simulate_knockout(group_winners["D"], group_runners_up["C"]), # Fra vs Pol
        simulate_knockout(group_winners["B"], group_runners_up["A"]), # Eng vs Sen
        simulate_knockout(group_winners["E"], group_runners_up["F"]), # Jap vs Cro
        simulate_knockout(group_winners["G"], group_runners_up["H"]), # Bra vs Kor
        simulate_knockout(group_winners["F"], group_runners_up["E"]), # Mor vs Spa
        simulate_knockout(group_winners["H"], group_runners_up["G"]), # Por vs Sui
    ]

    # ── 3. QUARTER-FINALS ──
    qf_winners = [
        simulate_knockout(r16_winners[4], r16_winners[5]), # Croatia / Brazil path
        simulate_knockout(r16_winners[1], r16_winners[0]), # Argentina / Netherlands path
        simulate_knockout(r16_winners[6], r16_winners[7]), # Morocco / Portugal path
        simulate_knockout(r16_winners[3], r16_winners[2]), # England / France path
    ]

    # ── 4. SEMI-FINALS ──
    sf_winners = [
        simulate_knockout(qf_winners[0], qf_winners[1]), # Semi 1
        simulate_knockout(qf_winners[2], qf_winners[3]), # Semi 2
    ]

    # ── 5. FINAL ──
    champion = simulate_knockout(sf_winners[0], sf_winners[1])
    return champion


#  MONTE CARLO EXECUTION (10,000 RUNS) 
n_simulations = 10_000
win_counter   = Counter()

print(f"Starting {n_simulations:,} tournament simulations...")
for i in range(n_simulations):
    winner = simulate_tournament()
    win_counter[winner] += 1
    
    # Progress indicator printout
    if (i + 1) % 2000 == 0:
        print(f" Completed {i + 1:,} / {n_simulations:,} simulations...")

# final outcome stats into a pandas metrics report
results_sim = pd.DataFrame(win_counter.most_common(), columns=["team", "wins"])
results_sim["probability"] = (results_sim["wins"] / n_simulations) * 100

print("\n=== BACKTEST SIMULATION RESULTS ===")
print(results_sim.head(20))